# 1-3. LangChain 기본: Prompt · Few-shot · LCEL · Output Parser

## 학습 목표
- `common.get_chat_model()`으로 LLM을 불러와 `invoke` / `stream`으로 호출한다.
- `PromptTemplate` · `ChatPromptTemplate`으로 재사용 가능한 프롬프트를 만든다.
- Few-shot 예시로 모델에 포맷을 학습시킨다 (금융 뉴스 감성 분류).
- LCEL(`|`)로 `prompt | llm | parser` 파이프라인을 조립한다.
- `StrOutputParser` · `JsonOutputParser` · `PydanticOutputParser`로 **구조화된 출력**을 얻어 다운스트림 코드가 바로 사용하도록 만든다.

## 핵심 키워드
`invoke`, `stream`, `PromptTemplate`, `ChatPromptTemplate`, `FewShotChatMessagePromptTemplate`, `LCEL`, `RunnablePassthrough`, `RunnableParallel`, `PydanticOutputParser`, `JsonOutputParser`

## 0. 준비


> 💡 이 노트북의 모든 예제는 금융 도메인(뉴스·약관·공시·여신 상담)을 소재로 한다.

In [1]:
import sys
sys.path.insert(0, '../..')

from common import get_chat_model, provider_badge

print(provider_badge())

☁️ OpenAI | model=openai/gpt-4o-mini


In [4]:
# 가장 짧은 호출: 문자열을 바로 넘긴다.
llm = get_chat_model(temperature=0.0)
print(llm.invoke('한 줄로 답하세요: 전자금융거래법의 목적은?').content)

전자금융거래법의 목적은 전자금융거래의 안전성과 신뢰성을 확보하고 이용자의 권익을 보호하는 것입니다.


`ChatOpenAI.invoke()`는 `AIMessage`를 돌려준다. `.content`에 본문이, `.response_metadata`에 사용량·모델 이름이 담긴다. 내부적으로 문자열은 `[HumanMessage(content=...)]`로 감싸진다.

## 1. PromptTemplate — 재사용 가능한 템플릿

프롬프트를 문자열 포맷으로 매번 쓰면 오탈자·중복이 쌓인다. `PromptTemplate`은 **입력 변수 스키마**를 선언해 실수를 막는다.

In [5]:
from langchain_core.prompts import PromptTemplate

# from_template: 중괄호 플레이스홀더를 자동으로 입력 변수로 인식
analysis_tpl = PromptTemplate.from_template(
    '당신은 주식 애널리스트입니다. 아래 종목을 세 줄 이내로 분석하세요.\n'
    '- 종목: {ticker}\n'
    '- 주가 변동률(YTD): {ytd_return}%\n'
    '- 최근 이슈: {headline}\n'
    '답변은 한국어로, 리스크 / 기회 / 코멘트 순서로 작성합니다.'
)

print('input_variables:', analysis_tpl.input_variables)

formatted = analysis_tpl.format(
    ticker='005930 삼성전자',
    ytd_return=12.4,
    headline='HBM3E 12단 양산 경쟁 격화로 수급 영향이 예상됨',
)
print('\n--- rendered prompt ---')
print(formatted)

input_variables: ['headline', 'ticker', 'ytd_return']

--- rendered prompt ---
당신은 주식 애널리스트입니다. 아래 종목을 세 줄 이내로 분석하세요.
- 종목: 005930 삼성전자
- 주가 변동률(YTD): 12.4%
- 최근 이슈: HBM3E 12단 양산 경쟁 격화로 수급 영향이 예상됨
답변은 한국어로, 리스크 / 기회 / 코멘트 순서로 작성합니다.


In [7]:
# 바로 LLM에 넣어본다. .invoke()는 dict를 받아 모든 변수를 채운다.
response = llm.invoke(analysis_tpl.invoke({
    'ticker': '035420 NAVER',
    'ytd_return': -4.8,
    'headline': '광고 매출 성장 둔화와 AI 서비스 투자 비용 확대가 동시 진행',
}))
print(response.content)

- 리스크: 광고 매출 성장 둔화는 수익성에 부정적인 영향을 미칠 수 있으며, AI 서비스 투자 비용 확대는 단기적으로 재무 부담을 증가시킬 수 있습니다.
- 기회: AI 서비스의 성공적인 개발 및 상용화는 장기적으로 새로운 수익원을 창출할 수 있으며, 시장 점유율 확대의 기회가 될 수 있습니다.
- 코멘트: 현재 주가는 하락세를 보이고 있으나, AI 분야의 성장 가능성을 고려할 때 중장기적으로 긍정적인 전망을 유지할 수 있습니다.


> ⚠️ 오탈자 주의: `PromptTemplate`은 변수명이 하나라도 빠지거나 남으면 `KeyError` / `ValidationError`를 던져 실행 전에 잡아준다. 이게 raw f-string 대비 가장 큰 장점.

## 2. ChatPromptTemplate — system + human (+ ai)

대부분의 실무 프롬프트는 역할이 **두 개 이상**이다. `ChatPromptTemplate`은 메시지 리스트를 직접 선언한다.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

# 튜플 방식 — 가장 간결
loan_counselor = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 시중은행의 여신 상담 챗봇입니다. '
     '반드시 금융소비자보호법의 설명의무 원칙에 따라 금리·상환방식·중도상환수수료를 빠짐없이 언급합니다. '
     '조건이 부족하면 질문으로 되물어 사용자가 오해하지 않도록 안내하세요.'),
    ('human',
     '저는 {age}세 {job}입니다. 주담대 {amount}만원을 {period}년 상환 조건으로 받고 싶어요. '
     '금리가 어떻게 되는지 알려주세요.'),
])

msg = loan_counselor.format_messages(age=34, job='직장인', amount=30000, period=30)
for m in msg:
    print(f'[{m.type}] {m.content[:80]}...')

[system] 당신은 시중은행의 여신 상담 챗봇입니다. 반드시 금융소비자보호법의 설명의무 원칙에 따라 금리·상환방식·중도상환수수료를 빠짐없이 언급합니다. 조건...
[human] 저는 34세 직장인입니다. 주담대 30000만원을 30년 상환 조건으로 받고 싶어요. 금리가 어떻게 되는지 알려주세요....


In [11]:
# 직접 호출
print(llm.invoke(msg).content)

주택담보대출(주담대)의 금리는 여러 요인에 따라 달라질 수 있습니다. 일반적으로 대출 금리는 신용도, 대출 기간, 대출 금액, 그리고 시장 금리에 따라 결정됩니다. 현재 금리는 약 3%에서 5% 사이일 수 있지만, 정확한 금리는 은행의 정책에 따라 다를 수 있습니다.

상환 방식은 원리금 균등상환, 원금 균등상환, 만기 일시상환 등 여러 가지가 있습니다. 원리금 균등상환은 매달 같은 금액을 상환하는 방식이고, 원금 균등상환은 매달 원금을 일정하게 상환하는 방식입니다. 만기 일시상환은 대출 만기 시에 전액을 상환하는 방식입니다.

또한, 중도상환수수료가 발생할 수 있습니다. 중도상환수수료는 대출을 조기 상환할 경우 발생하는 수수료로, 대출 계약서에 명시된 조건에 따라 다를 수 있습니다.

더 구체적인 금리나 조건을 알고 싶으시면, 신용도나 대출을 받으실 은행에 대한 정보가 필요합니다. 어떤 정보를 제공해 주실 수 있나요?


### `MessagesPlaceholder`로 대화 히스토리 끼워넣기

챗봇은 과거 대화 로그를 시스템 메시지 뒤에 주입해야 한다. `MessagesPlaceholder`가 그 자리를 비워 둔다.

In [12]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_tpl = ChatPromptTemplate.from_messages([
    ('system', '당신은 금융 Q&A 챗봇입니다. 간결히 답하세요.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}'),
])

history = [
    HumanMessage(content='전세대출이 뭐예요?'),
    AIMessage(content='임차인이 전세보증금 마련을 위해 금융기관에서 받는 대출입니다.'),
]

resp = llm.invoke(chat_tpl.invoke({
    'history': history,
    'question': '그럼 중도상환수수료가 붙나요?',
}))
print(resp.content)

네, 전세대출의 중도상환 시 중도상환수수료가 발생할 수 있습니다. 대출 상품에 따라 다르니, 계약서나 금융기관에 확인하는 것이 좋습니다.


## 3. Few-shot 프롬프팅 — 예시로 포맷을 학습시키기

아무리 지시를 구체적으로 써도 LLM이 출력 **형식**을 틀리는 경우가 많다. 예시를 2~5개 주면 모델이 형식을 흉내 낸다.

### 소재: 금융 뉴스 한 문장 감성 분류
`긍정 / 부정 / 중립` + 한 줄 근거.

In [13]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# 각 예시는 같은 스키마의 (input, output) 쌍
examples = [
    {
        'news': '한국은행이 기준금리를 0.25%p 인하했다. 시장은 이를 환영하는 분위기.',
        'answer': '긍정 | 금리 인하가 기업 자금조달 비용을 낮춰 증시에 우호적',
    },
    {
        'news': '○○저축은행의 부동산 PF 연체율이 전분기 대비 3%p 상승한 12.4%를 기록했다.',
        'answer': '부정 | 연체율 급등은 저축은행 건전성 악화 신호',
    },
    {
        'news': '금융위원회가 가상자산 2단계 법안 초안을 공개했으며 의견 수렴을 진행한다.',
        'answer': '중립 | 초안·의견 수렴 단계로 확정된 영향은 아직 없음',
    },
]

# 각 예시를 어떤 메시지 형식으로 풀어넣을지 정의
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '뉴스: {news}'),
    ('ai', '{answer}'),
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 최종 프롬프트: system → few-shot → 실제 질의
final_tpl = ChatPromptTemplate.from_messages([
    ('system',
     '너는 금융 뉴스 감성 분류기다. 각 뉴스 한 문장에 대해 '
     '`긍정|부정|중립 | 한 줄 근거` 형식으로만 답하라. 다른 말은 하지 마라.'),
    few_shot,
    ('human', '뉴스: {news}'),
])

sample = '삼성전자가 분기 영업이익 10조원을 돌파하며 반도체 업황 회복을 시사했다.'
print(llm.invoke(final_tpl.invoke({'news': sample})).content)

긍정 | 영업이익 증가가 반도체 시장 회복을 나타냄


### Few-shot이 없을 때 vs 있을 때

형식 준수율을 체감해 보기 위해 같은 질의를 few-shot 없이 돌려보자.

In [16]:
no_shot = ChatPromptTemplate.from_messages([
    ('system',
     '금융 뉴스의 감성을 `긍정|부정|중립 | 한 줄 근거` 형식으로만 분류하라.'),
    ('human', '뉴스: {news}'),
])

for s in [
    '국제유가가 배럴당 5달러 하락하며 항공·정유주 혼조세.',
    '○○카드의 연체율이 6개월 연속 상승세를 보이고 있다.',
    '증권거래소가 차세대 원장시스템의 개통 일정을 공지했다.',
]:
    print('📰', s)
    print('  [no-shot ]', llm.invoke(no_shot.invoke({'news': s})).content.strip())
    print('  [few-shot]', llm.invoke(final_tpl.invoke({'news': s})).content.strip())
    print()

📰 국제유가가 배럴당 5달러 하락하며 항공·정유주 혼조세.
  [no-shot ] 중립 | 국제유가 하락 소식이 항공·정유주에 혼조세를 보이고 있어 긍정적이거나 부정적이지 않음.
  [few-shot] 중립 | 유가 하락이 다양한 산업에 복합적 영향을 미침

📰 ○○카드의 연체율이 6개월 연속 상승세를 보이고 있다.
  [no-shot ] 부정 | 연체율 상승은 금융 건전성에 부정적인 신호로 해석될 수 있다.
  [few-shot] 부정 | 연체율 상승은 카드사의 신용 위험 증가를 의미함

📰 증권거래소가 차세대 원장시스템의 개통 일정을 공지했다.
  [no-shot ] 중립 | 차세대 원장시스템의 개통 일정 공지는 정보 전달에 해당함.
  [few-shot] 중립 | 일정 공지는 정보 제공일 뿐, 긍정적 또는 부정적 영향 없음



no-shot은 종종 `이 뉴스는…` 같은 불필요한 서두를 달거나 줄바꿈이 들어가 뒷단 파서가 터진다. few-shot을 주면 모델이 예시 모양을 그대로 따라간다.

## 4. LCEL — `prompt | llm | parser` 파이프라인

지금까지는 각 단계를 개별로 호출했다. LCEL(LangChain Expression Language)은 **파이프(`|`)**로 단계들을 엮어 하나의 `Runnable`을 만든다.

같은 `Runnable` 인터페이스에 4개 메서드가 모두 있다: `invoke`, `stream`, `batch`, `ainvoke`.

In [17]:
from langchain_core.output_parsers import StrOutputParser

chain = final_tpl | llm | StrOutputParser()

# 1) 단건 호출
print('invoke:', chain.invoke({'news': '금감원이 ○○증권에 기관주의 조치를 결정했다.'}))
print()

# 2) 스트리밍
print('stream: ', end='', flush=True)
for chunk in chain.stream({'news': '한국거래소 KOSPI200 지수 개편으로 편입 종목 변경이 발표됐다.'}):
    print(chunk, end='', flush=True)
print()

# 3) 배치 (내부적으로 동시 실행 → 네트워크 왕복 시간 아낌)
batch_in = [
    {'news': '은행권 가계대출 증가세가 3개월 연속 둔화됐다.'},
    {'news': '원·달러 환율이 1,400원을 재돌파하며 수입물가 부담 우려가 커졌다.'},
]
print('\nbatch:')
for r in chain.batch(batch_in):
    print('  -', r)

invoke: 부정 | 기관주의 조치는 증권사의 경영에 부정적 신호

stream: 중립 | 지수 개편은 시장에 중립적인 영향을 미칠 수 있음

batch:
  - 부정 | 가계대출 둔화는 소비 위축을 시사할 수 있음
  - 부정 | 환율 상승은 수입물가 부담 증가를 의미함


`StrOutputParser`가 하는 일은 단순하다: `AIMessage.content`에서 문자열만 꺼낸다. 체인 끝에 달아 두면 `.invoke()` 반환 타입이 `AIMessage`에서 `str`로 바뀌어 다운스트림 처리가 깔끔해진다.

## 5. LCEL 고급 — `RunnablePassthrough` · `RunnableParallel`

실제 RAG·요약 파이프라인은 "입력을 쪼개서 여러 경로로 보내고, 결과를 다시 합친다"의 반복이다. 두 블록으로 충분하다.

- `RunnableParallel` (dict 형태) — 입력 하나를 여러 브랜치로 동시에 실행, 결과를 dict로 묶어 다음 단계에 넘긴다.
- `RunnablePassthrough` — 입력을 그대로 통과. `.assign(key=runnable)`로 새 키를 얹을 수 있다.


In [18]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# 금융 공시 한 문단을 받아 [요약 / 위험도 한 줄평 / 키워드 3개]를 동시에 뽑는 파이프라인
disclosure = (
    '○○은행은 자회사 ○○캐피탈의 부실채권(NPL) 1,200억원을 일괄 매각하기로 결정했다. '
    '매각 상대방은 사모펀드 컨소시엄이며, 매각가는 장부가 대비 38% 수준인 450억원이다. '
    '이번 매각으로 은행의 고정이하여신비율은 1.45%에서 1.12%로 개선될 전망이다.'
)

summary_tpl  = ChatPromptTemplate.from_template('다음 공시를 한 문장으로 요약하라.\n\n{doc}')
risk_tpl     = ChatPromptTemplate.from_template('다음 공시의 주주 관점 리스크를 한 줄로.\n\n{doc}')
keywords_tpl = ChatPromptTemplate.from_template('다음 공시의 핵심 키워드 3개를 콤마로.\n\n{doc}')

summary_chain  = summary_tpl  | llm | StrOutputParser()
risk_chain     = risk_tpl     | llm | StrOutputParser()
keywords_chain = keywords_tpl | llm | StrOutputParser()

# 병렬 실행 — 세 체인을 한 번에 (내부적으로 async)
parallel = RunnableParallel(
    summary=summary_chain,
    risk=risk_chain,
    keywords=keywords_chain,
)

result = parallel.invoke({'doc': disclosure})
for k, v in result.items():
    print(f'[{k}] {v}')

[summary] ○○은행은 자회사 ○○캐피탈의 1,200억원 규모 부실채권을 사모펀드 컨소시엄에 450억원에 매각하기로 결정했으며, 이로 인해 고정이하여신비율이 1.45%에서 1.12%로 개선될 전망이다.
[risk] ○○은행의 자회사 부실채권 매각은 단기적으로 고정이하여신비율을 개선하지만, 매각가가 장부가의 38%에 불과해 자산 가치 하락과 향후 수익성에 대한 우려를 초래할 수 있다.
[keywords] 부실채권, 매각, 고정이하여신비율


In [19]:
# RunnablePassthrough.assign — 입력을 유지하면서 새 키를 주입
# "문서를 받아 요약을 추가한 뒤 최종 평가에 넘긴다" 같은 2-단계 체인에 유용
eval_tpl = ChatPromptTemplate.from_template(
    '원문과 요약을 보고 요약의 사실성(1~5)을 점수와 사유로 매겨라.\n\n'
    '원문: {doc}\n요약: {summary}'
)

pipeline = (
    RunnablePassthrough.assign(summary=summary_chain)  # {doc} → {doc, summary}
    | eval_tpl
    | llm
    | StrOutputParser()
)
print(pipeline.invoke({'doc': disclosure}))

점수: 5

사유: 요약은 원문의 주요 내용을 정확하게 반영하고 있으며, 부실채권의 규모, 매각 상대방, 매각가, 그리고 고정이하여신비율의 변화에 대한 정보를 모두 포함하고 있습니다. 원문과 요약 간의 사실적 일치가 완벽하여, 요약의 사실성이 매우 높다고 평가됩니다.


> 💡 RAG(2-4)에서 `{'context': retriever, 'question': RunnablePassthrough()}` 패턴을 계속 본다. 그 때 여기서 배운 형태 그대로다.

## 6. Output Parser — 구조화된 출력

모델 출력을 **파이썬 객체**로 바로 받아야 뒷단 코드가 깔끔해진다. LangChain은 세 가지 접근을 제공한다.

| Parser | 특징 | 언제 쓰나 |
|---|---|---|
| `JsonOutputParser` | 스키마 없이 JSON만 요구 | 가벼운 key-value |
| `PydanticOutputParser` | 타입·설명을 Pydantic 모델로 정의, 검증 | 필드가 많고 타입 안전이 필요할 때 |
| `StructuredOutputParser` + `ResponseSchema` | Pydantic 없이 간단 스키마 | 레거시 코드·빠른 프로토타이핑 |

모든 파서는 공통 인터페이스를 갖는다:
- `.get_format_instructions()` → 프롬프트에 삽입할 형식 지시문을 뽑아준다.
- `.parse(text)` → 문자열을 파이썬 객체로 변환.

### 6-1. `PydanticOutputParser` — 금융 리포트 구조화

금융 뉴스 한 건에서 **티커 · 감성 · 영향 섹터 · 신뢰도 · 근거** 5개 필드를 뽑는다.

In [21]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class NewsAnalysis(BaseModel):
    ticker: str = Field(description='주요 언급 종목의 티커(6자리) 또는 N/A')
    sentiment: Literal['긍정', '부정', '중립'] = Field(description='전반 감성')
    sectors: list[str] = Field(description='영향을 받는 섹터 1~3개')
    confidence: float = Field(ge=0.0, le=1.0, description='감성 분류 확신도 (0~1)')
    rationale: str = Field(description='감성 판단의 한 줄 근거')


parser = PydanticOutputParser(pydantic_object=NewsAnalysis)

# 핵심: get_format_instructions()로 JSON 스키마 힌트를 자동 생성
print('--- format instructions ---')
print(parser.get_format_instructions()[:600], '...\n')

report_tpl = ChatPromptTemplate.from_messages([
    ('system',
     '너는 금융 뉴스 분석기다. 반드시 아래 형식으로만 답하라.\n\n{format_instructions}'),
    ('human', '뉴스: {news}'),
]).partial(format_instructions=parser.get_format_instructions())

chain = report_tpl | llm | parser
result: NewsAnalysis = chain.invoke({
    'news': '삼성전자(005930)가 3분기 영업이익 11조원을 기록하며 시장 컨센서스를 상회했다. '
            '반도체 부문의 HBM 판매 호조가 실적을 견인했다.',
})

print('parsed type :', type(result).__name__)
print('ticker      :', result.ticker)
print('sentiment   :', result.sentiment)
print('sectors     :', result.sectors)
print('confidence  :', result.confidence)
print('rationale   :', result.rationale)

--- format instructions ---
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ticker": {"description": "주요 언급 종목의 티커(6자리) 또는 N/A", "title": "Ticker", "type": "string"}, "sentiment": {"description": "전반 감성", "en ...

parsed type : NewsAnalysis
ticker      : 005930
sentiment   : 긍정
sectors     : ['반도체', 'IT']
confidence  : 0.9
rationale   : 삼성전자가 시장 컨센서스를 상회하는 영업이익을 기록하여 긍정적인 실적을 보였다.


`ChatPromptTemplate.partial(...)`은 여러 입력 변수 중 **일부만 먼저 채워** 부분 템플릿을 만든다. 포맷 지시문처럼 호출할 때마다 동일한 값은 partial로 고정해두면 `.invoke()` 인자가 깔끔해진다.

### 6-2. `JsonOutputParser` — 스키마 없이 JSON만

Pydantic 모델을 만들 만큼은 아닌 간단한 구조는 `JsonOutputParser`로 충분하다. Pydantic 모델을 주면 **형식 지시문 + 검증**까지 자동, 안 주면 자유 JSON이다.

In [22]:
from langchain_core.output_parsers import JsonOutputParser

# Pydantic 없이 가벼운 JSON
json_parser = JsonOutputParser()

kv_tpl = ChatPromptTemplate.from_messages([
    ('system',
     '대출 약관 조항 원문에서 `rate(연이자율, %) / method(상환방식) / penalty(중도상환수수료 %)`'
     '를 JSON으로 뽑아내라. 숫자는 숫자 타입으로.\n{format_instructions}'),
    ('human', '{clause}'),
]).partial(format_instructions=json_parser.get_format_instructions())

sample_clause = (
    '본 상품의 연이자율은 연 5.27%이며, 원리금균등상환 방식으로 납입합니다. '
    '중도상환 시 잔여 원금의 1.2%를 수수료로 부과합니다.'
)

chain = kv_tpl | llm | json_parser
out = chain.invoke({'clause': sample_clause})
print('type:', type(out).__name__)
print(out)
print('rate:', out['rate'], '→', type(out['rate']).__name__)

type: dict
{'rate': 5.27, 'method': '원리금균등상환', 'penalty': 1.2}
rate: 5.27 → float


> 💡 스트리밍 시 `JsonOutputParser`는 **부분 JSON**을 누적 파싱해준다 (`chain.stream(...)`). 긴 구조화 응답을 UI에 흘려 보여줄 때 유용.

In [ ]:
# 스트리밍 + 부분 파싱 데모 (<!-- optional -->)
print('streaming partial JSON:')
for partial in chain.stream({'clause': sample_clause}):
    print('  ', partial)

### 6-3. `StructuredOutputParser` — Pydantic 없이

Pydantic 없이 필드 설명만으로 간단히 스키마를 만든다. 빠른 프로토타입에는 이쪽이 덜 번거롭다.

In [23]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

schemas = [
    ResponseSchema(name='issuer', description='채권 발행기관'),
    ResponseSchema(name='maturity', description='만기 (YYYY-MM-DD 형식)'),
    ResponseSchema(name='coupon', description='표면금리 (%, 실수)'),
]
s_parser = StructuredOutputParser.from_response_schemas(schemas)

bond_tpl = ChatPromptTemplate.from_messages([
    ('system', '채권 공시 한 문장에서 지정된 필드를 추출하라.\n{fmt}'),
    ('human', '{text}'),
]).partial(fmt=s_parser.get_format_instructions())

sample = '한국전력공사는 2030년 5월 15일 만기, 표면금리 4.125%의 원화 회사채를 발행한다.'
print((bond_tpl | llm | s_parser).invoke({'text': sample}))

{'issuer': '한국전력공사', 'maturity': '2030-05-15', 'coupon': '4.125'}


## 7. 실전 패턴 — 병렬 추출 + 파이프라인 합치기

지금까지 배운 블록을 합쳐 "공시 한 건 → 구조화된 리포트" 파이프라인을 만든다.

In [24]:
# 1) 구조화된 분석 (Pydantic)
# 2) 같은 입력에 대한 일반 요약
# 을 병렬로 실행하고 하나의 dict로 합친다.

pipeline = RunnableParallel(
    summary=(summary_tpl | llm | StrOutputParser()),
    analysis=(report_tpl | llm | PydanticOutputParser(pydantic_object=NewsAnalysis)),
)

news_body = (
    'NAVER(035420)는 AI 검색 도입 지연과 광고 단가 하락이 겹치며 3분기 영업이익이 '
    '전년 동기 대비 18% 감소한 3,200억원에 그쳤다. 다만 커머스·핀테크 부문 매출은 두 자리 성장세를 이어갔다.'
)

# PydanticOutputParser 체인은 내부에서 {news, format_instructions}를 요구하므로
# 입력을 둘 다 포함하는 dict가 필요함에 유의.
bundle = pipeline.invoke({'doc': news_body, 'news': news_body})
print('--- summary ---')
print(bundle['summary'])
print('\n--- structured analysis ---')
print(bundle['analysis'].model_dump_json(indent=2))

--- summary ---
NAVER는 AI 검색 도입 지연과 광고 단가 하락으로 3분기 영업이익이 18% 감소한 3,200억원을 기록했지만, 커머스와 핀테크 부문은 두 자리 성장세를 유지했다.

--- structured analysis ---
{
  "ticker": "035420",
  "sentiment": "부정",
  "sectors": [
    "IT",
    "커머스",
    "핀테크"
  ],
  "confidence": 0.85,
  "rationale": "3분기 영업이익 감소와 광고 단가 하락으로 부정적인 결과가 나타났기 때문."
}



## 더 읽어보기
- [LangChain · Prompt Templates](https://python.langchain.com/docs/concepts/prompt_templates/)
- [LangChain · LCEL Interface](https://python.langchain.com/docs/concepts/lcel/)
- [LangChain · Output Parsers](https://python.langchain.com/docs/concepts/output_parsers/)
- [Pydantic v2 문서](https://docs.pydantic.dev/latest/)